# 📊 Benchmark End-to-End: Toàn bộ Pipeline Traffic RAG
**Mục tiêu:** Chạy toàn bộ luồng `AgenticTrafficChat` (Classify → Rewrite → Retrieve → Generate) trên bộ câu hỏi Ground Truth và đo lường tự động các chỉ số:

| Chỉ số | Ý nghĩa |
|--------|---------|
| **Routing Accuracy** | Router có phân loại đúng loại câu hỏi không? |
| **Retrieval Precision@5** | Trong top-5 kết quả có chứa điều luật đúng không? |
| **Generation Faithfulness** | Câu trả lời có trích dẫn đúng số Điều/NĐ không? |
| **Latency** | Thời gian phản hồi trung bình |

In [ ]:
import os, sys, csv, json, time, yaml
from dotenv import load_dotenv

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

BASE_RESEARCH = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

import google.generativeai as genai
from rank_bm25 import BM25Okapi
from source.core.config import Settings
from source.retrieval.hybrid_retriever import HybridRetriever

settings = Settings()
api_key = settings.api_key or os.getenv('API_KEY')
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.0-flash')

# Load retriever + BM25
retriever = HybridRetriever(settings=settings, collection_name="Traffic_Law_Hybrid")
CHUNKS_PATH = os.path.join(PROJECT_ROOT, 'Data', 'chunks', 'traffic_chunks.json')
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    retriever.corpus_chunks = json.load(f)
retriever.bm25 = BM25Okapi([retriever._tokenize(c['content']) for c in retriever.corpus_chunks])

# Load prompts
PROMPT_PATH = os.path.join(PROJECT_ROOT, 'source', 'core', 'traffic_prompts.yaml')
with open(PROMPT_PATH, 'r', encoding='utf-8') as f:
    prompts = yaml.safe_load(f)['prompts']

print(f"✅ Môi trường sẵn sàng! {len(retriever.corpus_chunks):,} chunks trong bộ nhớ")

## 1. Load bộ Ground Truth

In [ ]:
GT_PATH = os.path.join(BASE_RESEARCH, 'Eval_System', 'dataset', 'ground_truth_traffic.csv')

ground_truth = []
with open(GT_PATH, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        ground_truth.append(row)

print(f"✅ Đã tải {len(ground_truth)} câu hỏi ground truth")
print(f"\n📋 Phân phối danh mục:")
from collections import Counter
cats = Counter(r['category'] for r in ground_truth)
for cat, cnt in cats.most_common():
    print(f"   {cat}: {cnt}")

## 2. Định nghĩa các hàm gọi pipeline

In [ ]:
import numpy as np

def call_gemini_prompt(prompt_name: str, **kwargs) -> str:
    full_prompt = ""
    for msg in prompts[prompt_name]['messages']:
        content = msg['content'].format(**kwargs)
        full_prompt += f"{'SYSTEM' if msg['role'] == 'system' else 'USER'}:\n{content}\n\n"
    return model.generate_content(full_prompt).text.strip()

def bm25_search(query: str, top_k: int = 5):
    tokenized_q = retriever._tokenize(query)
    scores = retriever.bm25.get_scores(tokenized_q)
    if np.max(scores) > 0:
        scores = scores / np.max(scores)
    top_indices = scores.argsort()[-top_k:][::-1]
    return [{"chunk": retriever.corpus_chunks[i], "score": float(scores[i])} for i in top_indices]

def run_pipeline_e2e(query: str, use_rewrite: bool = True):
    """Chạy toàn bộ pipeline và trả về kết quả kèm metadata benchmark."""
    start_time = time.time()
    
    # Step 1: Classify
    category_raw = call_gemini_prompt('classify_query', query=query)
    category = next((c for c in category_raw if c in ['0','1','2']), '?')
    
    if category != '1':
        return {"query": query, "category": category, "retrieved_docs": [],
                "answer": "[Không phải câu hỏi luật]", "latency": time.time()-start_time}
    
    # Step 2: Query Rewriting
    if use_rewrite:
        rewrite_raw = call_gemini_prompt('query_generator', query=query, history='Trống')
        search_queries = [q.strip() for q in rewrite_raw.split('\n') if q.strip()][:3]
    else:
        search_queries = [query]
    
    # Step 3: Multi-Query Retrieval
    seen = set()
    all_results = []
    for sq in search_queries:
        for r in bm25_search(sq, top_k=3):
            key = r['chunk']['content'][:60]
            if key not in seen:
                seen.add(key)
                all_results.append(r)
    top_results = sorted(all_results, key=lambda x: x['score'], reverse=True)[:5]
    
    # Step 4: Build context
    context_str = ""
    for i, r in enumerate(top_results, 1):
        m = r['chunk']['metadata']
        context_str += f"[{i}] {m.get('ten_van_ban','Luật')} | {m.get('dieu','?')}\nNội dung: {r['chunk']['content']}\n\n"
    
    # Step 5: Generate Answer
    answer = call_gemini_prompt('traffic_qa', context=context_str, history='Trống', query=query)
    
    return {
        "query": query,
        "category": category,
        "search_queries": search_queries,
        "retrieved_docs": top_results,
        "answer": answer,
        "latency": time.time() - start_time
    }

print("✅ Các hàm pipeline E2E đã sẵn sàng!")

## 3. Chạy Benchmark

In [ ]:
print(f"🚀 Bắt đầu Benchmark End-to-End trên {len(ground_truth)} câu hỏi...")
print("   (Mỗi câu hỏi cần ~5-10 giây do gọi Gemini nhiều lần)\n")

eval_results = []
for i, gt_row in enumerate(ground_truth):
    query = gt_row['question']
    expected_article = gt_row['expected_article']
    expected_answer = gt_row['expected_answer']
    
    print(f"[{i+1:02d}/{len(ground_truth)}] Đang xử lý: '{query[:55]}'...")
    
    try:
        result = run_pipeline_e2e(query, use_rewrite=True)
    except Exception as e:
        print(f"   ⚠️  Lỗi: {e}")
        result = {"query": query, "category": "?", "retrieved_docs": [], "answer": "", "latency": 0}
    
    # Đo các chỉ số
    # 1. Router Accuracy
    routing_correct = result['category'] == '1'  # Tất cả ground truth đều là câu luật
    
    # 2. Retrieval Precision@5: Điều luật mong đợi có xuất hiện trong top-5 không?
    retrieved = result.get('retrieved_docs', [])
    p_at_5 = 0.0
    for r in retrieved[:5]:
        dieu_in_meta = r['chunk']['metadata'].get('dieu', '')
        dieu_in_content = r['chunk']['content']
        if expected_article.lower() in dieu_in_meta.lower() or expected_article.lower() in dieu_in_content.lower():
            p_at_5 = 1.0
            break
    
    # 3. Generation Faithfulness: Câu trả lời có chứa số Điều mong đợi không?
    answer = result.get('answer', '')
    faithful = expected_article.lower() in answer.lower() or expected_article.split()[-1] in answer
    
    eval_results.append({
        "question": query,
        "expected_article": expected_article,
        "category": gt_row['category'],
        "routing_correct": routing_correct,
        "precision_at_5": p_at_5,
        "faithful": faithful,
        "latency": result['latency'],
        "answer_snippet": answer[:200] if answer else ""
    })
    
    status = "✅" if (routing_correct and p_at_5 > 0) else "⚠️"
    print(f"   {status} Router={routing_correct} | P@5={p_at_5:.0f} | Faithful={faithful} | {result['latency']:.1f}s\n")

print("✅ Benchmark hoàn tất!")

## 4. Tổng kết kết quả

In [ ]:
from collections import defaultdict

print("\n" + "="*60)
print("📊 KẾT QUẢ BENCHMARK END-TO-END")
print("="*60)

n = len(eval_results)
routing_acc = sum(r['routing_correct'] for r in eval_results) / n
p_at_5_avg = sum(r['precision_at_5'] for r in eval_results) / n
faithfulness = sum(r['faithful'] for r in eval_results) / n
avg_latency = sum(r['latency'] for r in eval_results) / n

print(f"  📌 Số câu hỏi test         : {n}")
print(f"  🚦 Routing Accuracy         : {routing_acc:.1%}")
print(f"  🔍 Retrieval Precision@5    : {p_at_5_avg:.1%}")
print(f"  ✍️  Generation Faithfulness  : {faithfulness:.1%}")
print(f"  ⏱️  Latency Trung bình       : {avg_latency:.1f}s/câu")

# Phân tích theo danh mục
print(f"\n📋 Phân tích theo danh mục:")
cat_stats = defaultdict(lambda: {'total': 0, 'p_hit': 0})
for r in eval_results:
    cat_stats[r['category']]['total'] += 1
    cat_stats[r['category']]['p_hit'] += r['precision_at_5']

for cat, stats in cat_stats.items():
    prec = stats['p_hit'] / stats['total']
    bar = '█' * int(prec * 10)
    print(f"   {cat:20s}: P@5={prec:.1%}  {bar}")

In [ ]:
# Các câu trả lời sai (Precision@5 = 0)
failures = [r for r in eval_results if r['precision_at_5'] == 0]
if failures:
    print(f"\n❌ {len(failures)} câu hỏi RETRIEVAL MISS (không tìm thấy điều luật đúng):")
    for f in failures:
        print(f"  Q: '{f['question']}'")
        print(f"     Kỳ vọng: {f['expected_article']} | Danh mục: {f['category']}")
        print()
else:
    print("\n🎉 Tất cả câu hỏi đều được truy xuất đúng điều luật!")

# Export kết quả
output_path = os.path.join(BASE_RESEARCH, 'Eval_System', 'dataset', 'benchmark_results.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(eval_results, f, ensure_ascii=False, indent=2)
print(f"\n💾 Đã lưu kết quả chi tiết vào: {output_path}")

## 5. So sánh có Rewriting vs không Rewriting

In [ ]:
# Chạy lại không có rewriting để so sánh với kết quả trên
print("🔄 Đang chạy lại KHÔNG có Query Rewriting để so sánh...")

no_rewrite_results = []
for gt_row in ground_truth[:5]:  # Chỉ chạy 5 câu đầu để tiết kiệm thời gian
    query = gt_row['question']
    expected = gt_row['expected_article']
    
    result_no_rw = run_pipeline_e2e(query, use_rewrite=False)
    retrieved = result_no_rw.get('retrieved_docs', [])
    p5 = 0.0
    for r in retrieved[:5]:
        if expected.lower() in r['chunk']['metadata'].get('dieu','').lower():
            p5 = 1.0; break
    no_rewrite_results.append({'question': query, 'p5': p5})

# Lấy cùng 5 câu từ kết quả rewriting
rw_p5 = [eval_results[i]['precision_at_5'] for i in range(min(5, len(eval_results)))]
no_rw_p5 = [r['p5'] for r in no_rewrite_results]

print(f"\n{'='*60}")
print(f"{'Câu hỏi':<45} {'Không RW':>10} {'Có RW':>8}")
print(f"{'-'*60}")
for i in range(len(no_rewrite_results)):
    q = ground_truth[i]['question'][:43]
    no_rw = '✅' if no_rw_p5[i] > 0 else '❌'
    rw = '✅' if rw_p5[i] > 0 else '❌'
    print(f"{q:<45} {no_rw:>10} {rw:>8}")

avg_no_rw = sum(no_rw_p5) / len(no_rw_p5)
avg_rw = sum(rw_p5) / len(rw_p5)
print(f"\n  Trung bình P@5 Không Rewrite: {avg_no_rw:.1%}")
print(f"  Trung bình P@5 Có Rewrite   : {avg_rw:.1%}")
print(f"  {'→ Query Rewriting CẢI THIỆN kết quả! 🎉' if avg_rw >= avg_no_rw else '→ Query Rewriting KHÔNG cải thiện, cần xem lại prompt!'}")